# Phase 1: Data Collection

Before any model can be built, the underlying data pipeline must be designed, documented, and validated. This notebook covers source identification, collection methodology, data cleaning, and standardization — producing the canonical datasets used in all downstream analysis. Every data decision is documented here so the reasoning is reproducible.

## Oscar Context

This notebook assembles the full dataset for Oscar prediction modeling. Sources include precursor ceremony results, expert consensus aggregators (Gold Derby), prediction markets (Kalshi, Polymarket), and ground truth winners. The output is a set of standardized CSVs consumed by all downstream notebooks.

## 1. Source Inventory

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Source registry — describes every candidate input to the model
SOURCES = pd.DataFrame([
    {"source":       "gold_derby",
     "type":         "expert_aggregator",
     "years":        "2019–2026",
     "categories":   "all 23–24",
     "access":       "Wayback Machine HTML scrape",
     "format":       "percentage probability",
     "notes":        "Aggregates ~30 journalists + editors + ~13k users. No API; must scrape snapshots."},
    {"source":       "kalshi",
     "type":         "prediction_market",
     "years":        "2024–2026",
     "categories":   "major only (~8–12)",
     "access":       "REST API (docs.kalshi.com)",
     "format":       "OHLC candlesticks (yes_price = implied prob)",
     "notes":        "$2.3M vol 2024, $29.6M 2025, ~$120M 2026. No Oscar data prior to 2024."},
    {"source":       "polymarket",
     "type":         "prediction_market",
     "years":        "2023–2026",
     "categories":   "major only (~5–8)",
     "access":       "CLOB API (docs.polymarket.com)",
     "format":       "price history (token_id based)",
     "notes":        "US access restricted until late 2025. 12hr+ granularity for resolved markets."},
    {"source":       "precursor_ceremonies",
     "type":         "binary_outcome",
     "years":        "2019–2026",
     "categories":   "varies by guild",
     "access":       "Wikipedia HTML tables",
     "format":       "winner/nominee binary",
     "notes":        "DGA, SAG, PGA, BAFTA, Globes, Critics Choice, ASC, ACE Eddie, CDG, WGA."},
    {"source":       "oscar_ground_truth",
     "type":         "ground_truth",
     "years":        "1928–2026",
     "categories":   "all",
     "access":       "Kaggle CSV (theacademy/academy-awards)",
     "format":       "winner/nominee binary",
     "notes":        "Canonical target variable. Download manually from Kaggle."},
])

display(SOURCES.set_index("source"))
print(f"\nData will be written to: {RAW_DIR.resolve()}")


## 2. Precursor Ceremony Results

## 3. Expert Consensus Aggregator (Gold Derby)

Gold Derby aggregates predictions from ~30 credentialed journalists, in-house editors,
and up to 13,000 users. It publishes percentage win probabilities for all 23–24 Oscar
categories. No API exists; data must be recovered from Wayback Machine snapshots of
their live odds pages (`goldderby.com/odds/oscar-odds/<category>/`).

**Step 1:** Query the Wayback Machine CDX API to build a snapshot inventory — what
years and categories are actually archived, and how close to each ceremony date.

**Step 2:** For each available (ceremony, category) pair, fetch and parse the snapshot.

**Step 3:** Run QC on the resulting dataset.


In [ ]:
from src.goldderby import CATEGORY_SLUGS, build_snapshot_inventory
from src.wayback import CEREMONY_DATES

# --- Step 1: Enumerate available Wayback Machine snapshots ----------------
# This queries the CDX API for every Gold Derby Oscar odds category page.
# It takes several minutes — results are cached to disk.

SNAPSHOT_INVENTORY_PATH = RAW_DIR / "goldderby_snapshot_inventory.csv"

if SNAPSHOT_INVENTORY_PATH.exists():
    print("Loading cached snapshot inventory...")
    inventory = pd.read_csv(SNAPSHOT_INVENTORY_PATH)
else:
    print("Querying Wayback Machine CDX API (this will take a few minutes)...")
    inventory = build_snapshot_inventory()
    inventory.to_csv(SNAPSHOT_INVENTORY_PATH, index=False)
    print(f"Saved to {SNAPSHOT_INVENTORY_PATH}")

# Coverage summary: how many (ceremony × category) pairs have snapshots
total = len(inventory)
available = inventory["snapshot_ts"].notna().sum()
missing = total - available

print(f"\nSnapshot inventory: {available}/{total} pairs have archived snapshots "
      f"({100*available/total:.1f}% coverage)")
display(
    inventory.pivot_table(
        index="category",
        columns="ceremony",
        values="days_before_ceremony",
        aggfunc="first"
    ).style.format("{:.0f}", na_rep="—")
     .set_caption("Days before ceremony for each archived snapshot (— = no snapshot found)")
)


In [ ]:
from src.goldderby import scrape_all, CATEGORY_SLUGS
from src.qc import run_all as qc_run_all

# --- Step 2: Scrape all available snapshots --------------------------------
# Set FORCE_RESCRAPE = True to re-fetch even if the cached file exists.
FORCE_RESCRAPE = False
GD_RAW_PATH = RAW_DIR / "goldderby_raw.csv"

if GD_RAW_PATH.exists() and not FORCE_RESCRAPE:
    print("Loading cached Gold Derby data...")
    gd_raw = pd.read_csv(GD_RAW_PATH)
else:
    # To scrape only specific ceremonies or categories, pass lists here:
    #   scrape_all(ceremonies=[96, 97, 98], categories=["best-picture", "best-director"])
    gd_raw = scrape_all()
    gd_raw["source"] = "gold_derby"
    gd_raw.to_csv(GD_RAW_PATH, index=False)
    print(f"Saved {len(gd_raw)} rows to {GD_RAW_PATH}")

print(f"\nRaw Gold Derby data: {len(gd_raw):,} rows")
print(f"  Ceremonies covered: {sorted(gd_raw['ceremony'].unique())}")
print(f"  Categories covered: {gd_raw['category'].nunique()} of {len(CATEGORY_SLUGS)}")
print(f"  Predictor groups:   {sorted(gd_raw['group'].unique())}")
display(gd_raw.head(10))


In [ ]:
from src.qc import run_all as qc_run_all, ALL_CATEGORIES, EXPECTED_CEREMONIES

# --- Step 3: QC on raw Gold Derby data ------------------------------------
# Run all 12 checks. Failures are informational at this stage —
# we annotate problems rather than dropping data.

gd_report = qc_run_all(
    gd_raw,
    source_label="Gold Derby (raw)",
    expected_ceremonies=EXPECTED_CEREMONIES,
    expected_categories=ALL_CATEGORIES,
    show_detail=True,
)

# Persist the QC summary
gd_report.summary_df().to_csv(RAW_DIR / "qc_goldderby_raw.csv", index=False)
print("QC summary saved.")


## 4. Prediction Markets (Kalshi, Polymarket)

Both platforms expose public REST APIs for historical market prices.

**Kalshi:** `GET /historical/markets/{ticker}/candlesticks`  
Markets exist under the `KXOSCARS` series (confirm via `GET /series`).
Data available from 2024 onwards. Candlestick period = 1440 min (daily) for pre-ceremony snapshot.

**Polymarket:** `GET /prices-history?token_id=<id>`  
Oscar events accessible via Gamma API by slug (e.g. `oscars-best-picture`).
US-restricted until late 2025; resolved market granularity limited to 12hr+.

For both: we want the final pre-ceremony closing price (= implied win probability).


In [ ]:
import requests
import time
from src.wayback import _get_with_retry

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

# --- Step 1: Enumerate all Oscar series markets ---------------------------
# We'll look for markets under any series with "oscar" in the title.

def kalshi_get(path: str, params: dict = None) -> dict:
    url = KALSHI_BASE + path
    resp = _get_with_retry(url, params=params,
                           headers={"Accept": "application/json"})
    return resp.json()

# Discover Oscar series tickers
series_resp = kalshi_get("/series", params={"limit": 200})
oscar_series = [s for s in series_resp.get("series", [])
                if "oscar" in s.get("title", "").lower()
                or "oscar" in s.get("ticker", "").lower()]
print(f"Found {len(oscar_series)} Oscar-related series:")
for s in oscar_series:
    print(f"  ticker={s['ticker']}  title={s.get('title','')}")


In [ ]:
from datetime import datetime, timedelta

# Standardized category slug → Kalshi market ticker fragment mapping.
# Tickers confirmed from Kalshi's 2025/2026 Oscar hub.
# Format: KXOSCARS-<YEAR>-<SLUG> (exact tickers confirmed at scrape time)
KALSHI_CATEGORY_MAP = {
    "best-picture":            "BP",
    "best-director":           "BD",
    "best-actor":              "BAC",
    "best-actress":            "BAC",   # disambiguate by full ticker
    "best-supporting-actor":   "BSA",
    "best-supporting-actress": "BSAC",
    "best-animated-feature":   "BAF",
    "best-documentary-feature":"BDF",
    "best-international-film": "BIF",
    "best-adapted-screenplay": "BAS",
    "best-original-screenplay":"BOS",
    "best-original-score":     "BOSC",
    "best-original-song":      "BOSN",
    "best-cinematography":     "BC",
    "best-film-editing":       "BFE",
}

def kalshi_fetch_final_price(series_ticker: str, market_ticker: str,
                              ceremony_date: str) -> float | None:
    """
    Fetch the last closing price (yes_price) before a given ceremony date
    from Kalshi's historical candlestick endpoint. Returns implied probability.
    """
    ceremony_dt = datetime.strptime(ceremony_date, "%Y%m%d")
    start_ts = int((ceremony_dt - timedelta(days=7)).timestamp() * 1000)
    end_ts   = int(ceremony_dt.timestamp() * 1000)
    try:
        # First try historical endpoint (settled markets)
        data = kalshi_get(f"/historical/markets/{market_ticker}/candlesticks",
                          params={"period_interval": 1440,
                                  "start_ts": start_ts, "end_ts": end_ts})
        candles = data.get("candlesticks", [])
        if candles:
            last = sorted(candles, key=lambda c: c["end_period_ts"])[-1]
            return last["yes"]["close"] / 100.0  # Kalshi prices are in cents
    except Exception as e:
        print(f"  Historical endpoint failed for {market_ticker}: {e}")
    # Fallback: live endpoint
    try:
        data = kalshi_get(f"/series/{series_ticker}/markets/{market_ticker}/candlesticks",
                          params={"period_interval": 1440,
                                  "start_ts": start_ts, "end_ts": end_ts})
        candles = data.get("candlesticks", [])
        if candles:
            last = sorted(candles, key=lambda c: c["end_period_ts"])[-1]
            return last["yes"]["close"] / 100.0
    except Exception as e:
        print(f"  Live endpoint also failed for {market_ticker}: {e}")
    return None


def scrape_kalshi_oscars(ceremonies: list[int] = None) -> pd.DataFrame:
    """
    For each Oscar ceremony and each Kalshi-covered category, fetch the
    final pre-ceremony probability. Returns tidy DataFrame.
    """
    from src.wayback import CEREMONY_DATES
    ceremonies = ceremonies or [96, 97, 98]  # Kalshi years: 2024, 2025, 2026
    records = []

    for ceremony in ceremonies:
        date = CEREMONY_DATES[ceremony]
        year = int(date[:4])
        # Enumerate all markets in the Oscar series for this year
        try:
            mkts_resp = kalshi_get("/markets",
                                    params={"series_ticker": f"KXOSCARS",
                                            "status": "settled", "limit": 200})
            all_markets = mkts_resp.get("markets", [])
        except Exception as e:
            print(f"  Could not list markets for ceremony {ceremony}: {e}")
            continue

        # Filter to this ceremony year
        year_markets = [m for m in all_markets if str(year) in m.get("ticker", "")]
        print(f"  Ceremony {ceremony} ({year}): {len(year_markets)} markets found")

        for mkt in year_markets:
            ticker = mkt["ticker"]
            category_guess = _guess_category_from_ticker(ticker)
            nominee = mkt.get("subtitle", mkt.get("title", ""))

            # Try to get the final price from candlestick history
            prob = kalshi_fetch_final_price("KXOSCARS", ticker, date)

            # Fallback: use the settlement result (1.0 = won, 0.0 = lost)
            if prob is None and mkt.get("status") == "settled":
                result = mkt.get("result")
                if result == "yes":
                    prob = 1.0
                elif result == "no":
                    prob = 0.0

            if prob is not None:
                records.append({
                    "ceremony":    ceremony,
                    "year":        year,
                    "category":    category_guess,
                    "nominee":     nominee,
                    "group":       "market",
                    "probability": prob,
                    "source":      "kalshi",
                    "ticker":      ticker,
                })
    return pd.DataFrame(records)


def _guess_category_from_ticker(ticker: str) -> str:
    """Heuristic: map a Kalshi ticker to our standardized category slug."""
    ticker_upper = ticker.upper()
    for slug, fragment in KALSHI_CATEGORY_MAP.items():
        if fragment in ticker_upper:
            return slug
    return "unknown"


# ---- Run the Kalshi scrape -----------------------------------------------
KALSHI_RAW_PATH = RAW_DIR / "kalshi_raw.csv"
FORCE_RESCRAPE = False

if KALSHI_RAW_PATH.exists() and not FORCE_RESCRAPE:
    print("Loading cached Kalshi data...")
    kalshi_raw = pd.read_csv(KALSHI_RAW_PATH)
else:
    kalshi_raw = scrape_kalshi_oscars(ceremonies=[96, 97, 98])
    kalshi_raw.to_csv(KALSHI_RAW_PATH, index=False)

print(f"\nKalshi raw: {len(kalshi_raw):,} rows")
if not kalshi_raw.empty:
    display(kalshi_raw.head(10))


In [ ]:
from src.qc import run_all as qc_run_all, MAJOR_CATEGORIES

# QC on Kalshi — only major categories, ceremonies 96–98
if not kalshi_raw.empty:
    kalshi_report = qc_run_all(
        kalshi_raw,
        source_label="Kalshi (raw)",
        expected_ceremonies=[96, 97, 98],
        expected_categories=MAJOR_CATEGORIES,
    )
    kalshi_report.summary_df().to_csv(RAW_DIR / "qc_kalshi_raw.csv", index=False)
else:
    print("No Kalshi data to QC — check API connectivity.")


## 5. Ground Truth: Actual Winners

The canonical target variable. Source: Kaggle `theacademy/academy-awards` dataset.

**Manual step required:** download `the_oscar_award.csv` from
[kaggle.com/datasets/theacademy/academy-awards](https://www.kaggle.com/datasets/theacademy/academy-awards)
and place it at `data/raw/the_oscar_award.csv`.


In [ ]:
GROUND_TRUTH_PATH = RAW_DIR / "the_oscar_award.csv"

# Kaggle column names: year_film, year_ceremony, ceremony, category, name, film, winner
CATEGORY_NAME_MAP = {
    "BEST PICTURE":                          "best-picture",
    "DIRECTING":                             "best-director",
    "ACTOR IN A LEADING ROLE":               "best-actor",
    "ACTRESS IN A LEADING ROLE":             "best-actress",
    "ACTOR IN A SUPPORTING ROLE":            "best-supporting-actor",
    "ACTRESS IN A SUPPORTING ROLE":          "best-supporting-actress",
    "WRITING (ADAPTED SCREENPLAY)":          "best-adapted-screenplay",
    "WRITING (ORIGINAL SCREENPLAY)":         "best-original-screenplay",
    "ANIMATED FEATURE FILM":                 "best-animated-feature",
    "DOCUMENTARY FEATURE FILM":              "best-documentary-feature",
    "DOCUMENTARY (FEATURE)":                 "best-documentary-feature",
    "INTERNATIONAL FEATURE FILM":            "best-international-film",
    "FOREIGN LANGUAGE FILM":                 "best-international-film",
    "CINEMATOGRAPHY":                        "best-cinematography",
    "FILM EDITING":                          "best-film-editing",
    "MUSIC (ORIGINAL SCORE)":                "best-original-score",
    "MUSIC (ORIGINAL SONG)":                 "best-original-song",
    "PRODUCTION DESIGN":                     "best-production-design",
    "COSTUME DESIGN":                        "best-costume-design",
    "MAKEUP AND HAIRSTYLING":                "best-makeup-hairstyling",
    "SOUND":                                 "best-sound",
    "VISUAL EFFECTS":                        "best-visual-effects",
    "SHORT FILM (DOCUMENTARY)":              "best-documentary-short",
    "DOCUMENTARY SHORT FILM":               "best-documentary-short",
    "SHORT FILM (ANIMATED)":                 "best-animated-short",
    "ANIMATED SHORT FILM":                   "best-animated-short",
    "SHORT FILM (LIVE ACTION)":              "best-live-action-short",
    "LIVE ACTION SHORT FILM":                "best-live-action-short",
    "CASTING":                               "best-casting",
}

if not GROUND_TRUTH_PATH.exists():
    print(f"⚠  Ground truth file not found at {GROUND_TRUTH_PATH}")
    print("   Download from: https://www.kaggle.com/datasets/theacademy/academy-awards")
    print("   Place as: data/raw/the_oscar_award.csv")
    ground_truth = pd.DataFrame()
else:
    raw_gt = pd.read_csv(GROUND_TRUTH_PATH)

    # Normalize and filter to ceremonies 91–98
    gt = raw_gt.copy()
    gt.columns = [c.lower().strip() for c in gt.columns]
    gt["category_slug"] = gt["category"].str.upper().str.strip().map(CATEGORY_NAME_MAP)
    gt = gt.dropna(subset=["category_slug"])
    gt = gt[gt["ceremony"].between(91, 98)]
    gt["winner"] = gt["winner"].astype(bool)

    # Use the nominee's film as the label (consistent with Gold Derby / Kalshi)
    gt["nominee"] = gt["film"].fillna(gt["name"]).str.strip()

    ground_truth = gt[["ceremony", "category_slug", "nominee", "winner"]].rename(
        columns={"category_slug": "category"}
    ).copy()

    print(f"Ground truth: {len(ground_truth):,} nominees across "
          f"{ground_truth['ceremony'].nunique()} ceremonies and "
          f"{ground_truth['category'].nunique()} categories")
    ground_truth.to_csv(RAW_DIR / "ground_truth.csv", index=False)
    display(ground_truth[ground_truth["winner"]].head(15))


## 6. Data Quality Audit

All sources have already been checked individually above. This section:

1. Merges all sources into a single combined DataFrame
2. Joins winner labels from ground truth
3. Runs the full QC suite on the combined dataset — including cross-source agreement
   and frontrunner sanity checks that require ground truth
4. Emits a `qc_combined_report.csv` for reference in notebook 02


In [ ]:
from src.qc import run_all as qc_run_all, ALL_CATEGORIES, EXPECTED_CEREMONIES

# --- 6.1  Combine all probability sources ---------------------------------
frames = []
for label, df in [("gold_derby", gd_raw if "gd_raw" in dir() else pd.DataFrame()),
                   ("kalshi",     kalshi_raw if "kalshi_raw" in dir() else pd.DataFrame())]:
    if not df.empty:
        df = df.copy()
        df["source"] = label
        frames.append(df)

if not frames:
    raise RuntimeError("No data loaded. Run sections 3 and 4 first.")

combined = pd.concat(frames, ignore_index=True)

# Ensure dtypes are correct for QC
combined["ceremony"]    = pd.to_numeric(combined["ceremony"], errors="coerce").astype("Int64")
combined["year"]        = pd.to_numeric(combined["year"],     errors="coerce").astype("Int64")
combined["probability"] = pd.to_numeric(combined["probability"], errors="coerce")

print(f"Combined dataset: {len(combined):,} rows")
print(combined.groupby(["source", "ceremony"])["category"].nunique()
      .rename("categories").reset_index().to_string(index=False))


In [ ]:
# --- 6.2  Join winner labels from ground truth ----------------------------
# This enables frontrunner_sanity and winner_coverage QC checks.

if not ground_truth.empty:
    # Fuzzy nominee name matching is hard; use exact join first,
    # flag unmatched rows as a QC finding.
    combined = combined.merge(
        ground_truth[["ceremony", "category", "nominee", "winner"]],
        on=["ceremony", "category", "nominee"],
        how="left",
    )
    matched = combined["winner"].notna().sum()
    total = len(combined)
    print(f"Winner join: {matched:,}/{total:,} rows matched ({100*matched/total:.1f}%)")
    print("Unmatched nominee examples (name mismatch between sources):")
    unmatched = combined[combined["winner"].isna()][["source","ceremony","category","nominee"]].drop_duplicates()
    display(unmatched.head(20))
else:
    combined["winner"] = np.nan
    print("Ground truth not available — winner column will be NaN (skipping winner QC checks)")


In [ ]:
# --- 6.3  Master QC run on combined dataset --------------------------------
combined_report = qc_run_all(
    combined,
    source_label="All Sources Combined",
    expected_ceremonies=EXPECTED_CEREMONIES,
    expected_categories=ALL_CATEGORIES,
    show_detail=True,
)
combined_report.summary_df().to_csv(RAW_DIR / "qc_combined_report.csv", index=False)

# Surface any critical failures
critical_failures = [r for r in combined_report.results
                     if not r.passed and r.severity == "critical"]
if critical_failures:
    print("\n⚠  CRITICAL QC FAILURES — do not proceed to notebook 02 without resolving:")
    for f in critical_failures:
        print(f"   {f.check_name}: {f.message}")
else:
    print("\n✓  No critical failures. Data is safe to use in downstream notebooks.")


## 7. Output: Standardized CSVs

In [ ]:
# Final processed outputs consumed by notebooks 02–06.
# Conventions:
#   - All probabilities in [0, 1]
#   - Nominee names are the film title where applicable (not the person)
#   - Category slugs match CATEGORY_SLUGS in src/goldderby.py
#   - group="combined" is the primary row for ensemble modeling;
#     group="experts"|"editors"|"users" rows are retained for audit

# --- probability_sources.csv: one row per (ceremony, category, nominee, group, source) ---
prob_cols = ["ceremony", "year", "category", "nominee", "group", "source", "probability"]
optional = ["winner", "snapshot_ts"]
out_cols = prob_cols + [c for c in optional if c in combined.columns]
prob_out = combined[out_cols].copy()
prob_out = prob_out.sort_values(["ceremony", "category", "source", "group", "probability"],
                                 ascending=[True, True, True, True, False])
prob_out.to_csv(PROCESSED_DIR / "probability_sources.csv", index=False)
print(f"probability_sources.csv: {len(prob_out):,} rows → {PROCESSED_DIR}/probability_sources.csv")

# --- ground_truth.csv: one row per (ceremony, category, nominee) ----------
if not ground_truth.empty:
    ground_truth.to_csv(PROCESSED_DIR / "ground_truth.csv", index=False)
    print(f"ground_truth.csv:         {len(ground_truth):,} rows")

# --- coverage_summary.csv: what (ceremony × category × source) triplets exist ---
coverage = (combined
    .groupby(["ceremony", "category", "source"])
    .agg(n_nominees=("nominee", "nunique"),
         top_prob=("probability", "max"),
         has_winner_label=("winner", lambda x: x.notna().any()))
    .reset_index())
coverage.to_csv(PROCESSED_DIR / "coverage_summary.csv", index=False)
print(f"coverage_summary.csv:     {len(coverage):,} rows")

print("\nAll outputs written to data/processed/")
display(coverage.head(20))
